# Required Setup — Neo4j Credentials

**Run this notebook once before starting any other notebooks.**

This notebook stores your Neo4j Aura connection details and both Genie Space IDs as Databricks secrets
so all subsequent notebooks can connect without re-entering credentials.

The Delta Lake tables (accounts, account_labels, merchants, transactions, account_links)
were already loaded by your workshop admin.

### Prerequisites

- A running **Neo4j Aura** instance with connection details ready
- A **Dedicated** compute cluster with the Neo4j Spark Connector installed:
  - Access mode: **Dedicated** (required for the Spark Connector)
  - Runtime: 13.3 LTS or higher
  - Maven library: `org.neo4j:neo4j-connector-apache-spark_2.12:5.3.1_for_spark_3`

## Step 1: Enter Your Connection Details

Fill in the five constants in the cell below:

- **NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD** — your Neo4j Aura connection details
- **GENIE_SPACE_ID_BEFORE** — the ID of the Genie Space pointed at the base tables (before graph enrichment).
- **GENIE_SPACE_ID_AFTER** — the ID of the Genie Space pointed at the enriched gold tables (after graph enrichment).

Find each space ID in the Databricks UI under **Genie → your space → the ID in the URL** (e.g. `01ef1234abcd5678`).

All five values will be stored as Databricks secrets in scope `neo4j-graph-engineering`.

In [ ]:
NEO4J_URI      = 'NEO4J_URI'
NEO4J_USERNAME = 'neo4j_username'
NEO4J_PASSWORD = 'neo4j_password'

GENIE_SPACE_ID_BEFORE = 'YOUR-BEFORE-SPACE-ID'   # e.g. '01ef1234abcd5678'
GENIE_SPACE_ID_AFTER  = 'YOUR-AFTER-SPACE-ID'    # e.g. '01ef1234abcd5679'

if NEO4J_URI == 'NEO4J_URI' or NEO4J_PASSWORD == 'neo4j_password':
    raise ValueError(
        'Please replace the NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD placeholders before running.'
    )

if GENIE_SPACE_ID_BEFORE == 'YOUR-BEFORE-SPACE-ID':
    raise ValueError(
        'Please replace GENIE_SPACE_ID_BEFORE with your before-enrichment Genie Space ID. '
        'Find it in the Databricks UI under Genie → your space → the ID in the URL.'
    )

if GENIE_SPACE_ID_AFTER == 'YOUR-AFTER-SPACE-ID':
    raise ValueError(
        'Please replace GENIE_SPACE_ID_AFTER with your after-enrichment Genie Space ID. '
        'Find it in the Databricks UI under Genie → your space → the ID in the URL.'
    )

## Step 2: Store Credentials and Verify Connection

The cells below create the secret scope `neo4j-graph-engineering` (if it does not
already exist), store your credentials, then verify the connection is live.

In [ ]:
from databricks.sdk import WorkspaceClient

SCOPE = 'neo4j-graph-engineering'
w = WorkspaceClient()

print(f'Creating secret scope: {SCOPE}')
try:
    w.secrets.create_scope(scope=SCOPE)
    print(f'  [OK] Scope {SCOPE!r} created')
except Exception as e:
    if 'already exists' in str(e).lower():
        print(f'  [OK] Scope {SCOPE!r} already exists')
    else:
        raise

secrets = {
    'uri':                  NEO4J_URI,
    'username':             NEO4J_USERNAME,
    'password':             NEO4J_PASSWORD,
    'genie_space_id_before': GENIE_SPACE_ID_BEFORE,
    'genie_space_id_after':  GENIE_SPACE_ID_AFTER,
}
for key, value in secrets.items():
    w.secrets.put_secret(scope=SCOPE, key=key, string_value=value)
    print(f'  [OK] {key} stored')

In [ ]:
%pip install neo4j --quiet

In [ ]:
from neo4j import GraphDatabase

print(f'Verifying Neo4j connection: {NEO4J_URI}')
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session(database='neo4j') as session:
        record = session.run("RETURN 'Connected' AS status").single()
        print(f'  [OK] Neo4j responded: {record["status"]}')
    driver.close()
except Exception as e:
    print(f'  [FAIL] Could not connect to Neo4j: {e}')
    print('  Check that the URI starts with neo4j+s:// for Aura and credentials are correct.')
    raise

## Next Steps

If all steps above show `[OK]`, proceed to **`01_genie_before`** to run the structural-discovery
questions against the base tables and confirm the gap that graph enrichment will close.